# SMS Spam & Phishing Detection with BERT

Fine-tuning `bert-base-uncased` to classify SMS messages as **ham** (legitimate)
or **spam** (unsolicited / phishing).

This notebook is **stage 1 of a two-model security pipeline**:

```
   ┌─────────────────────────┐      spam + contains URL      ┌──────────────────────────┐
   │  SMS Spam Detection      │  ──────────────────────────▶ │  Malicious URL Detection │
   │  (BERT, this project)    │                              │  (CharCNN, companion repo)│
   └─────────────────────────┘                              └──────────────────────────┘
```

A message is first screened for spam/phishing. When it is flagged **and** a URL
is present, the raw link is handed off to the companion
[`malicious-url-detector`](https://github.com/johnsonchiang26-dev/malicious-url-detector)
(a character-level CNN) for deeper analysis — mirroring how a real SMS
abuse-detection pipeline triages messages then escalates the embedded payload.

**Walkthrough**
1. Exploratory Data Analysis
2. Text Preprocessing
3. Train / Val / Test split
4. Baseline — TF-IDF + Logistic Regression
5. BERT fine-tuning (hand-written training loop)
6. Test-set evaluation (Baseline vs BERT)
7. Error analysis
8. Live inference demo

## 0. Setup & imports

We run everything from the **project root** so that relative paths (`data/`,
`docs/`, `checkpoints/`) resolve, and add the repo root to `sys.path` so the
reusable `src` package is importable.

In [ ]:
import os
import sys

# Run from the project root regardless of where Jupyter was launched.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

from src.utils import clean_sms_text, load_config, set_seed, LABEL2ID, ID2LABEL

config = load_config("configs/default.yaml")
set_seed(config["seed"])
os.makedirs("docs", exist_ok=True)
config

## 1. Exploratory Data Analysis

First we make sure the data is present (downloading the UCI corpus on first run)
and load it together with our curated financial-phishing examples.

In [ ]:
from data.download_data import download_uci, write_phishing_examples
from src.train import load_dataframe

# Curated phishing CSV is offline data — always (re)written. UCI is downloaded once.
write_phishing_examples()
if not os.path.exists(config["data_path"]):
    download_uci()

df = load_dataframe(config["data_path"], config["phishing_path"])
print("total messages:", len(df))
df.head()

### 1.1 Class distribution

The corpus is **imbalanced** (~87% ham / ~13% spam). This is why we later report
F1 / precision / recall for the spam class rather than relying on accuracy, and
why we select the best checkpoint by **validation F1**.

In [ ]:
counts = df["label"].value_counts()
pct = (counts / len(df) * 100).round(2)
print(pd.DataFrame({"count": counts, "percent": pct}))

fig, ax = plt.subplots(figsize=(5, 4))
counts.loc[["ham", "spam"]].plot(kind="bar", color=["#4C9F70", "#D7263D"], ax=ax)
ax.set_title("Class distribution")
ax.set_xlabel("")
ax.set_ylabel("messages")
ax.tick_params(axis="x", rotation=0)
for i, v in enumerate(counts.loc[["ham", "spam"]]):
    ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.savefig("docs/class_distribution.png", bbox_inches="tight")
plt.show()

### 1.2 Message length

Spam messages tend to be **longer** than ham — they pack in calls to action,
links and money amounts. This is a useful sanity signal (and an implicit feature
BERT can exploit through positional/length cues).

In [ ]:
df["n_chars"] = df["text"].str.len()
df["n_words"] = df["text"].str.split().apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes, ["n_chars", "n_words"], ["Character count", "Word count"]):
    for label, color in [("ham", "#4C9F70"), ("spam", "#D7263D")]:
        ax.hist(df.loc[df["label"] == label, col], bins=40, alpha=0.6, label=label, color=color)
    ax.set_title(f"{title} by class")
    ax.set_xlabel(col)
    ax.legend()
plt.tight_layout()
plt.savefig("docs/message_length_distribution.png", bbox_inches="tight")
plt.show()

df.groupby("label")[["n_chars", "n_words"]].agg(["mean", "median", "max"]).round(1)

### 1.3 Sample messages

A look at raw ham, raw spam, and our **curated financial-phishing** rows (the
bridge to real-world security scenarios).

In [ ]:
print("=== HAM (legit) ===")
for t in df[df["label"] == "ham"]["text"].head(3):
    print(" •", t)

print("\n=== SPAM (UCI) ===")
for t in df[(df["label"] == "spam") & (df["source"] == "uci")]["text"].head(3):
    print(" •", t)

print("\n=== CURATED FINANCIAL PHISHING ===")
for t in df[df["source"] == "phishing"]["text"].head(5):
    print(" •", t)

## 2. Text preprocessing

`clean_sms_text()` **normalises** spam signals into typed placeholders rather
than deleting them — the *presence* of a link / phone / money amount is itself
predictive, while the specific value (a one-off URL) is noise the model should
not memorise:

| pattern | placeholder |
|---|---|
| URLs (`http(s)://`, `www.`, bare domains) | `[URL]` |
| phone numbers | `[PHONE]` |
| emails | `[EMAIL]` |
| money (`$ £ € ¥` + amounts) | `[MONEY]` |

Plus lowercasing and whitespace collapsing.

In [ ]:
samples = [
    "WINNER!! You've been selected for a £1000 prize. Claim at http://bit.ly/win-now",
    "Your KYC is incomplete. Verify within 24h: www.bank-kyc-verify.tk/update",
    "Crypto withdrawal of $4,800 pending. Authorise: http://wallet-confirm.app/auth",
    "Hey, call me on 07700 900123 when you're free",
    "Ok lah, see you at the cafe later",
]
pd.DataFrame({
    "before": samples,
    "after": [clean_sms_text(s) for s in samples],
})

## 3. Train / Val / Test split

A **70 / 10 / 20** split, **stratified** by label so every split keeps the same
ham:spam ratio — essential under class imbalance.

In [ ]:
from src.train import stratified_split

splits = stratified_split(df, config["train_ratio"], config["val_ratio"], config["seed"])

for name, (texts, labels) in splits.items():
    labels = np.array(labels)
    print(f"{name:5s}: {len(labels):5d} messages | spam = {labels.mean():.1%}")

## 4. Baseline — TF-IDF + Logistic Regression

Before reaching for BERT, we establish a strong, cheap classical baseline. This
demonstrates the model isn't deep-learning-for-its-own-sake: BERT has to *earn*
its place by beating a well-tuned bag-of-words model.

We use word 1–2 grams on the **cleaned** text, and `class_weight="balanced"` to
counter the imbalance.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from src.evaluate import compute_metrics

Xtr_text = [clean_sms_text(t) for t in splits["train"][0]]
Xte_text = [clean_sms_text(t) for t in splits["test"][0]]
y_train = np.array(splits["train"][1])
y_test = np.array(splits["test"][1])

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
Xtr = tfidf.fit_transform(Xtr_text)
Xte = tfidf.transform(Xte_text)

baseline = LogisticRegression(max_iter=1000, class_weight="balanced")
baseline.fit(Xtr, y_train)

base_pred = baseline.predict(Xte)
base_prob = baseline.predict_proba(Xte)[:, 1]
baseline_metrics = compute_metrics(y_test, base_pred, base_prob)
print("Baseline (TF-IDF + LogReg) test metrics:")
for k, v in baseline_metrics.items():
    print(f"  {k:10s}: {v:.4f}")

## 5. BERT fine-tuning

Now the main event. The training loop is written **by hand** (no
`transformers.Trainer`) to make every moving part explicit:

- **AdamW** with weight decay excluded from bias & LayerNorm parameters
- **Linear warmup** over the first 10% of steps, then linear decay
- **Gradient clipping** at max-norm 1.0
- **Early stopping** on validation F1 (patience = 2)
- **Best-checkpoint saving** by validation F1

> On CPU this is slow; a single GPU (e.g. Colab T4) finishes in a few minutes.

In [ ]:
import torch
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from src.dataset import create_dataloaders
from src.model import BertSMSClassifier
from src.train import build_optimizer, get_device, train_one_epoch
from src.evaluate import predict_loader

device = get_device()
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(config["model_name"])
loaders = create_dataloaders(
    splits, tokenizer,
    max_length=config["max_length"],
    batch_size=config["batch_size"],
)

model = BertSMSClassifier(
    model_name=config["model_name"],
    num_labels=config["num_labels"],
    dropout=config["dropout"],
).to(device)

optimizer = build_optimizer(model, config["learning_rate"], config["weight_decay"])
total_steps = len(loaders["train"]) * config["epochs"]
warmup_steps = int(config["warmup_ratio"] * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print(f"total optimisation steps: {total_steps} (warmup {warmup_steps})")

In [ ]:
best_f1, epochs_without_improvement = -1.0, 0
os.makedirs(config["output_dir"], exist_ok=True)
history = []

for epoch in range(1, config["epochs"] + 1):
    train_loss = train_one_epoch(
        model, loaders["train"], optimizer, scheduler, device, config["gradient_clip"]
    )
    yt, yp, ypr = predict_loader(model, loaders["val"], device)
    val_metrics = compute_metrics(yt, yp, ypr)
    history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})
    print(f"epoch {epoch}/{config['epochs']}  "
          f"train_loss={train_loss:.4f}  "
          f"val_f1={val_metrics['f1']:.4f}  val_acc={val_metrics['accuracy']:.4f}")

    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]
        epochs_without_improvement = 0
        model.save_pretrained(config["output_dir"])
        tokenizer.save_pretrained(config["output_dir"])
        print(f"   ↳ new best val F1 = {best_f1:.4f} — checkpoint saved")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= config["early_stopping_patience"]:
            print(f"early stopping at epoch {epoch}")
            break

pd.DataFrame(history)

## 6. Test-set evaluation — Baseline vs BERT

We reload the **best** checkpoint (selected on validation F1) and evaluate it on
the held-out test set, then compare head-to-head with the baseline.

In [ ]:
from sklearn.metrics import classification_report

best_model = BertSMSClassifier.from_pretrained(config["output_dir"]).to(device)
y_true, bert_pred, bert_prob = predict_loader(best_model, loaders["test"], device)
bert_metrics = compute_metrics(y_true, bert_pred, bert_prob)

print("=== BERT — classification report ===")
print(classification_report(y_true, bert_pred, target_names=["ham", "spam"], digits=4))

In [ ]:
results = pd.DataFrame(
    [baseline_metrics, bert_metrics],
    index=["TF-IDF + LogReg", "BERT"],
)[["accuracy", "precision", "recall", "f1", "f1_macro", "roc_auc"]]
print("Spam-class metrics (precision/recall/f1) + overall accuracy & ROC-AUC:")
results.round(4)

### 6.1 Confusion matrices

Side-by-side confusion matrices make the **error trade-off** concrete: BERT's
advantage is typically in the bottom-left cell (fewer missed spam — higher
recall), which is what matters most for a security filter.

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, preds) in zip(axes, [("TF-IDF + LogReg", base_pred), ("BERT", bert_pred)]):
    cm = confusion_matrix(y_test if name.startswith("TF") else y_true, preds, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["ham", "spam"], yticklabels=["ham", "spam"], ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("docs/confusion_matrices.png", bbox_inches="tight")
plt.show()

### 6.2 ROC curves

In [ ]:
from src.evaluate import plot_roc_curves

plot_roc_curves(
    {
        "TF-IDF + LogReg": (y_test, base_prob),
        "BERT": (y_true, bert_prob),
    },
    save_path="docs/roc_curves.png",
)
plt.show()

## 7. Error analysis

Which messages does BERT get wrong, and how confident is it when it does? We
extract **false positives** (legit blocked) and **false negatives** (spam missed)
with their model confidence — the rows worth inspecting before any real
deployment.

In [ ]:
from src.evaluate import error_analysis

test_texts = splits["test"][0]
errors = error_analysis(test_texts, y_true, bert_pred, bert_prob)
print(f"total errors: {len(errors)}  "
      f"(FP={sum(errors['error_type'] == 'FP')}, FN={sum(errors['error_type'] == 'FN')})")
errors.head(15)

In [ ]:
print("False NEGATIVES (spam the model missed) — most confident first:")
fn = errors[errors["error_type"] == "FN"]
display(fn.head(10)) if "display" in dir() else print(fn.head(10).to_string())

print("\nFalse POSITIVES (ham the model wrongly flagged):")
fp = errors[errors["error_type"] == "FP"]
display(fp.head(10)) if "display" in dir() else print(fp.head(10).to_string())

## 8. Live inference demo

Finally, the end-to-end inference path via the `SMSClassifier` wrapper, on six
fresh messages (3 spam/phishing incl. financial, 3 ham). When a message is
flagged spam **and** contains a URL, the classifier surfaces the raw link — the
**hand-off point** to the `malicious-url-detector` (CharCNN) stage.

In [ ]:
from src.predict import SMSClassifier

clf = SMSClassifier(model_dir=config["output_dir"], max_length=config["max_length"])

demo_messages = [
    # --- spam / phishing ---
    "URGENT: Your bank account has been locked. Verify now: http://secure-verify-account.com/login",
    "Congratulations! You won a $1000 gift card. Claim before midnight: http://bit.ly/claim-now",
    "Your crypto withdrawal of $4,800 is pending approval. Authorise it: http://wallet-confirm.app/auth",
    # --- ham ---
    "Hey, are we still meeting for lunch at 1pm tomorrow?",
    "Your OTP for the transaction of GBP 45.00 at TESCO is 552190. Do not share this code with anyone.",
    "Don't forget to pick up milk on your way home :)",
]

for r in clf.predict_batch(demo_messages):
    flag = "🚨 SPAM" if r["label"] == "spam" else "✅ HAM "
    print(f"{flag} | p_spam={r['p_spam']:.3f} | {r['text'][:70]}")
    if r["label"] == "spam" and r["urls"]:
        print(f"         └─▶ forward URL(s) to malicious-url-detector: {r['urls']}")
    print()

## Summary & next steps

- A hand-written BERT fine-tuning pipeline beats a strong TF-IDF + LogReg
  baseline on SMS spam/phishing, with the largest gain in **spam recall** — the
  metric that matters for an abuse filter.
- Preprocessing **normalises** (does not delete) spam signals into typed
  placeholders, so the model learns robust structural cues instead of memorising
  one-off URLs.
- Best checkpoint is chosen by **validation F1** under class imbalance, and
  error analysis surfaces the residual FP/FN for review.

**Pipeline next stage** → spam messages that carry a URL are escalated to the
companion [`malicious-url-detector`](https://github.com/johnsonchiang26-dev/malicious-url-detector)
(character-level CNN) for link-level threat scoring:

```
SMS spam detection  →  spam contains a URL  →  CharCNN URL maliciousness scan
```